In [1]:
import sys, torch, os
print("Python", sys.version.split()[0], "| Torch", torch.__version__, "| CUDA", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")
!nvidia-smi --query-gpu=memory.total,memory.used --format=csv 2>/dev/null || true
!free -g | head -2

Python 3.11.13 | Torch 2.6.0+cu124 | CUDA True
GPU: NVIDIA A100-SXM4-40GB 42 GB
memory.total [MiB], memory.used [MiB]
40960 MiB, 6 MiB
               total        used        free      shared  buff/cache   available
Mem:              83           3          75           0           4          79


In [2]:
!pip install -q onnx onnxruntime onnxscript jiwer unidecode h5py pyyaml flashlight-text huggingface_hub pandas praat-textgrids
from torchaudio.models.decoder import ctc_decoder
print("torchaudio ctc_decoder import OK")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 115.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.0/412.0 kB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 144.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<

In [3]:
from google.colab import drive
drive.mount("/content/drive")
import os, sys, glob, shutil
# ===================== EDIT THESE =====================
DRIVE_PROJECT = "/content/drive/MyDrive/silent_speech"        # your project folder on Drive
GADDY_CKPT    = f"{DRIVE_PROJECT}/gaddy_recognition_model.pt"  # your 54M Gaddy checkpoint
# =====================================================
REPO_DIR = "/content/silent_speech"
LM_DIR   = os.path.join(REPO_DIR, "KenLM")
OUT_DIR  = f"{DRIVE_PROJECT}/compressed"          # every model is saved here
os.makedirs(OUT_DIR, exist_ok=True)
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/MatteoFasulo/silent_speech.git {REPO_DIR}
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
print("cwd:", os.getcwd(), "| OUT_DIR:", OUT_DIR)

Mounted at /content/drive
Cloning into '/content/silent_speech'...
remote: Enumerating objects: 496, done.
remote: Counting objects: 100% (341/341), done.
remote: Compressing objects: 100% (191/191), done.
remote: Total 496 (delta 229), reused 227 (delta 142), pack-reused 155 (from 1)
Receiving objects: 100% (496/496), 7.72 MiB | 18.65 MiB/s, done.
Resolving deltas: 100% (298/298), done.
cwd: /content/silent_speech | OUT_DIR: /content/drive/MyDrive/silent_speech/compressed


In [4]:
import yaml
H5_REAL = f"{DRIVE_PROJECT}/emg_dataset.h5"
if not os.path.exists(H5_REAL):
    c = sorted(glob.glob("/content/drive/MyDrive/**/emg_dataset.h5", recursive=True)); assert c, "emg_dataset.h5 not found"
    H5_REAL = c[0]
for cfg in glob.glob(os.path.join(REPO_DIR, "config", "*.y*ml")):
    d = yaml.safe_load(open(cfg)) or {}
    if isinstance(d, dict) and "h5_path" in d:
        d["h5_path"] = H5_REAL; yaml.safe_dump(d, open(cfg, "w"), sort_keys=False)
print("h5:", H5_REAL)
os.makedirs(LM_DIR, exist_ok=True)
lm = glob.glob("/content/drive/MyDrive/**/lm.bin", recursive=True) + glob.glob(f"{REPO_DIR}/**/lm.bin", recursive=True)
assert lm, "lm.bin not found"
if os.path.abspath(lm[0]) != os.path.abspath(os.path.join(LM_DIR,"lm.bin")): shutil.copy(lm[0], os.path.join(LM_DIR,"lm.bin"))
lex = glob.glob(f"{REPO_DIR}/**/gaddy_derived_lexicon.txt", recursive=True); assert lex, "lexicon not found"
if os.path.abspath(lex[0]) != os.path.abspath(os.path.join(LM_DIR,"gaddy_derived_lexicon.txt")): shutil.copy(lex[0], os.path.join(LM_DIR,"gaddy_derived_lexicon.txt"))
print("KenLM dir:", os.listdir(LM_DIR))
if not os.path.exists(GADDY_CKPT):
    import requests
    print("fetching checkpoint from Zenodo 7183877 ...")
    rec = requests.get("https://zenodo.org/api/records/7183877").json()
    url = [f for f in rec["files"] if f["key"].endswith(".pt")][0]["links"]["self"]
    os.makedirs(os.path.dirname(GADDY_CKPT), exist_ok=True)
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(GADDY_CKPT,"wb") as fo:
            for ch in r.iter_content(1<<20): fo.write(ch)
print("checkpoint:", GADDY_CKPT, f"{os.path.getsize(GADDY_CKPT)/1e6:.1f} MB")

h5: /content/drive/MyDrive/silent_speech/emg_dataset.h5
KenLM dir: ['gaddy_derived_lexicon.txt', 'lm.bin']
checkpoint: /content/drive/MyDrive/silent_speech/gaddy_recognition_model.pt 216.6 MB


In [18]:
# ===== Self-contained Gaddy recognition model (keys identical to the checkpoint) =====
import copy, random
import torch
from torch import nn
import torch.nn.functional as F
from typing import Optional

D_MODEL, N_HEAD, N_LAYERS, FFN, DROPOUT, RPD = 768, 8, 6, 3072, 0.0, 100

class LearnedRelativePositionalEmbedding(nn.Module):
    def __init__(self, max_relative_pos, num_heads, embedding_dim, unmasked=False,
                 heads_share_embeddings=False, add_to_values=False):
        super().__init__()
        self.max_relative_pos = max_relative_pos; self.num_heads = num_heads
        self.embedding_dim = embedding_dim; self.unmasked = unmasked
        self.heads_share_embeddings = heads_share_embeddings; self.add_to_values = add_to_values
        num_embeddings = 2 * max_relative_pos - 1 if unmasked else max_relative_pos
        embedding_size = [num_embeddings, embedding_dim, 1] if heads_share_embeddings \
            else [num_heads, num_embeddings, embedding_dim, 1]
        if add_to_values: embedding_size[-1] = 2
        initial_stddev = embedding_dim**(-0.5)
        self.embeddings = nn.Parameter(torch.zeros(*embedding_size))
        nn.init.normal_(self.embeddings, mean=0.0, std=initial_stddev)
    def forward(self, query, saved_state=None):
        length = query.shape[0]; decoder_step = False
        used = self.get_embeddings_for_query(length)
        vals = used[..., 1] if self.add_to_values else None
        pl = self.calculate_positional_logits(query, used[..., 0])
        pl = self.relative_to_absolute_indexing(pl, decoder_step)
        return (pl, vals)
    def get_embeddings_for_query(self, length):
        pad_length = max(length - self.max_relative_pos, 0)
        start_pos = max(self.max_relative_pos - length, 0)
        if self.unmasked:
            with torch.no_grad():
                padded = nn.functional.pad(self.embeddings, (0,0,0,0,pad_length,pad_length))
            used = padded.narrow(-3, start_pos, 2*length-1)
        else:
            with torch.no_grad():
                padded = nn.functional.pad(self.embeddings, (0,0,0,0,pad_length,0))
            used = padded.narrow(-3, start_pos, length)
        return used
    def calculate_positional_logits(self, query, relative_embeddings):
        if self.heads_share_embeddings:
            pl = torch.einsum("lbd,md->lbm", query, relative_embeddings)
        else:
            query = query.view(query.shape[0], -1, self.num_heads, self.embedding_dim)
            pl = torch.einsum("lbhd,hmd->lbhm", query, relative_embeddings)
            pl = pl.contiguous().view(pl.shape[0], -1, pl.shape[-1])
        length = query.size(0)
        if length > self.max_relative_pos:
            pad_length = length - self.max_relative_pos
            pl[:,:,:pad_length] -= 1e8
            if self.unmasked: pl[:,:,-pad_length:] -= 1e8
        return pl
    def relative_to_absolute_indexing(self, x, decoder_step):
        length, bsz_heads, _ = x.shape
        if decoder_step: return x.contiguous().view(bsz_heads, 1, -1)
        if self.unmasked:
            x = nn.functional.pad(x, (0,1)); x = x.transpose(0,1)
            x = x.contiguous().view(bsz_heads, length*2*length)
            x = nn.functional.pad(x, (0, length-1))
            x = x.view(bsz_heads, length+1, 2*length-1)
            return x[:, :length, length-1:]
        else:
            x = nn.functional.pad(x, (1,0)); x = x.transpose(0,1)
            x = x.contiguous().view(bsz_heads, length+1, length)
            return x[:, 1:, :]

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=256, n_head=4, dropout=0.1, relative_positional=True,
                 relative_positional_distance=100):
        super().__init__()
        self.d_model = d_model; self.n_head = n_head
        d_qkv = d_model // n_head
        assert d_qkv * n_head == d_model
        self.d_qkv = d_qkv
        self.w_q = nn.Parameter(torch.Tensor(n_head, d_model, d_qkv))
        self.w_k = nn.Parameter(torch.Tensor(n_head, d_model, d_qkv))
        self.w_v = nn.Parameter(torch.Tensor(n_head, d_model, d_qkv))
        self.w_o = nn.Parameter(torch.Tensor(n_head, d_qkv, d_model))
        nn.init.xavier_normal_(self.w_q); nn.init.xavier_normal_(self.w_k)
        nn.init.xavier_normal_(self.w_v); nn.init.xavier_normal_(self.w_o)
        self.dropout = nn.Dropout(dropout)
        if relative_positional:
            self.relative_positional = LearnedRelativePositionalEmbedding(
                relative_positional_distance, n_head, d_qkv, True)
        else:
            self.relative_positional = None
    def forward(self, x):
        q = torch.einsum('tbf,hfa->bhta', x, self.w_q)
        k = torch.einsum('tbf,hfa->bhta', x, self.w_k)
        v = torch.einsum('tbf,hfa->bhta', x, self.w_v)
        logits = torch.einsum('bhqa,bhka->bhqk', q, k) / (self.d_qkv ** 0.5)
        if self.relative_positional is not None:
            q_pos = q.permute(2,0,1,3)
            l,b,h,d = q_pos.size()
            position_logits, _ = self.relative_positional(q_pos.reshape(l,b*h,d))
            logits = logits + position_logits.view(b,h,l,l)
        probs = F.softmax(logits, dim=-1); probs = self.dropout(probs)
        o = torch.einsum('bhqk,bhka->bhqa', probs, v)
        out = torch.einsum('bhta,haf->tbf', o, self.w_o)
        return out

class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1,
                 relative_positional=True, relative_positional_distance=100):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, nhead, dropout=dropout,
            relative_positional=relative_positional, relative_positional_distance=relative_positional_distance)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model); self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout); self.dropout2 = nn.Dropout(dropout)
        self.activation = nn.ReLU()
    def forward(self, src, src_mask=None, src_key_padding_mask=None, is_causal=False):
        src2 = self.self_attn(src)
        src = src + self.dropout1(src2); src = self.norm1(src)
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src))))
        src = src + self.dropout2(src2); src = self.norm2(src)
        return src

class SimpleEncoder(nn.Module):
    """Key-identical, modern-torch-safe replacement for nn.TransformerEncoder."""
    def __init__(self, layer, n, norm=None):
        super().__init__()
        self.layers = nn.ModuleList([copy.deepcopy(layer) for _ in range(n)])
    def forward(self, x, mask=None, src_key_padding_mask=None, is_causal=None):
        for l in self.layers: x = l(x)
        return x

class ResBlock(nn.Module):
    def __init__(self, num_ins, num_outs, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(num_ins, num_outs, 3, padding=1, stride=stride)
        self.bn1 = nn.BatchNorm1d(num_outs)
        self.conv2 = nn.Conv1d(num_outs, num_outs, 3, padding=1)
        self.bn2 = nn.BatchNorm1d(num_outs)
        if stride != 1 or num_ins != num_outs:
            self.residual_path = nn.Conv1d(num_ins, num_outs, 1, stride=stride)
            self.res_norm = nn.BatchNorm1d(num_outs)
        else:
            self.residual_path = None
    def forward(self, x):
        iv = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        res = self.res_norm(self.residual_path(iv)) if self.residual_path is not None else iv
        return F.relu(x + res)

class Model(nn.Module):
    def __init__(self, num_features, num_outs, num_aux_outs=None,
                 model_size=D_MODEL, num_layers=N_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.conv_blocks = nn.Sequential(
            ResBlock(8, model_size, 2),
            ResBlock(model_size, model_size, 2),
            ResBlock(model_size, model_size, 2),
        )
        self.w_raw_in = nn.Linear(model_size, model_size)
        layer = TransformerEncoderLayer(d_model=model_size, nhead=N_HEAD, relative_positional=True,
            relative_positional_distance=RPD, dim_feedforward=FFN, dropout=dropout)
        self.transformer = SimpleEncoder(layer, num_layers)
        self.w_out = nn.Linear(model_size, num_outs)
        self.has_aux_out = num_aux_outs is not None
        if self.has_aux_out:
            self.w_aux = nn.Linear(model_size, num_aux_outs)
    def forward(self, x_feat, x_raw, session_ids):
        if self.training:
            r = random.randrange(8)
            if r > 0:
                shifted = torch.zeros_like(x_raw)
                shifted[:, :-r, :] = x_raw[:, r:, :]
                x_raw = shifted
        x_raw = x_raw.transpose(1,2)
        x_raw = self.conv_blocks(x_raw)
        x_raw = x_raw.transpose(1,2)
        x_raw = self.w_raw_in(x_raw)
        x = x_raw.transpose(0,1)
        x = self.transformer(x)
        x = x.transpose(0,1)
        if self.has_aux_out:
            return self.w_out(x), self.w_aux(x)
        return self.w_out(x)


# ---- build + load ----
import os
from hdf5_dataset import H5EmgDataset
_probe = H5EmgDataset(dev=False, test=True)
N_CHARS = len(_probe.text_transform.chars); NUM_OUTS = N_CHARS + 1
print("n_chars:", N_CHARS, "| num_outs:", NUM_OUTS, "(expect 37 / 38)")
model = Model(num_features=0, num_outs=NUM_OUTS).eval()
sd = torch.load(GADDY_CKPT, map_location="cpu", weights_only=False)
if isinstance(sd, dict) and "model_state_dict" in sd: sd = sd["model_state_dict"]
missing, unexpected = model.load_state_dict(sd, strict=False)
print("missing:", len(missing), "| unexpected:", len(unexpected))
assert len(missing)==0, ("checkpoint mismatch:", missing[:8])
print("params:", sum(p.numel() for p in model.parameters()))

n_chars: 37 | num_outs: 38 (expect 37 / 38)
missing: 0 | unexpected: 0
params: 54117926


In [6]:
import numpy as np, tqdm, jiwer, copy
import torch.nn.functional as F
from torchaudio.models.decoder import ctc_decoder

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(os.cpu_count() or 4)
BEAM = 1500                       # lower (e.g. 150) = much faster eval, slightly worse WER
print("DEVICE:", DEVICE, "| CPU threads:", torch.get_num_threads(), "| beam:", BEAM)

testset = _probe
# cache all (raw_emg, target) in RAM once (you have plenty)
TEST = []
for i in range(len(testset)):
    ex = testset[i]; tgt = testset.text_transform.clean_text(ex["text"])
    if tgt != "": TEST.append((ex["raw_emg"], tgt))
print("cached test utterances:", len(TEST))

def build_decoder(dset, beam=BEAM, lm_weight=4.0, word_score=-1.75):
    tkns = [c for c in dset.text_transform.chars] + ["_"]
    return ctc_decoder(lexicon=os.path.join(LM_DIR,"gaddy_derived_lexicon.txt"), tokens=tkns,
                       lm=os.path.join(LM_DIR,"lm.bin"), blank_token="_", sil_token="|",
                       nbest=1, lm_weight=lm_weight, word_score=word_score, sil_score=0.0, beam_size=beam)
DECODER = build_decoder(testset)

def _wer(get_logp, desc=""):
    refs, preds = [], []
    for raw, tgt in tqdm.tqdm(TEST, desc=desc):
        logp = get_logp(raw)                             # (1, T, 38) cpu
        pred = testset.text_transform.clean_text(" ".join(DECODER(logp)[0][0].words).strip())
        refs.append(tgt); preds.append(pred)
    return jiwer.wer(refs, preds)

def wer_torch(m, device=DEVICE, half=False, desc="torch"):
    m = m.eval().to(device); dummy = torch.zeros(1,1,dtype=torch.long,device=device)
    def gl(raw):
        x = raw.unsqueeze(0).to(device)
        if half: x = x.half()
        with torch.no_grad(): y = m(x, x, dummy)
        return F.log_softmax(y.float().cpu(), dim=-1)
    return _wer(gl, desc)

def wer_onnx(path, desc="onnx"):
    import onnxruntime as ort
    s = ort.InferenceSession(path, providers=["CPUExecutionProvider"]); nm = s.get_inputs()[0].name
    def gl(raw):
        y = s.run(None, {nm: raw.numpy().astype(np.float32)[None,...]})[0]
        return F.log_softmax(torch.from_numpy(y), dim=-1)
    return _wer(gl, desc)

def sd_mb(m): torch.save(m.state_dict(), "/content/_t.pt"); return os.path.getsize("/content/_t.pt")/1e6
def file_mb(p): return os.path.getsize(p)/1e6
def save_torch(m, fn):
    p=os.path.join(OUT_DIR,fn); torch.save(m.state_dict(),p); print("   saved ->",p,f"({os.path.getsize(p)/1e6:.1f} MB)"); return p
def save_file(src, fn):
    p=os.path.join(OUT_DIR,fn); shutil.copy(src,p); print("   saved ->",p,f"({os.path.getsize(p)/1e6:.1f} MB)"); return p

RESULTS = []
def log_result(name, wer, size_mb):
    RESULTS.append({"method":name,"WER%":round(wer*100,2),"size_MB":round(size_mb,2)})
    print(f"  -> {name:26s} WER {wer*100:6.2f}%  size {size_mb:8.2f} MB")
print("harness ready")

DEVICE: cuda | CPU threads: 12 | beam: 1500
cached test utterances: 98
harness ready


In [7]:
# ===== BASELINE FP32 (non-compressed) — evaluated on GPU, saved to Drive =====
BASE_WER = wer_torch(copy.deepcopy(model), device=DEVICE, desc="FP32 baseline")
BASE_SIZE = sd_mb(model)
log_result("Baseline FP32 (torch)", BASE_WER, BASE_SIZE)
save_torch(model, "gaddy_fp32.pt")
print(f"\nBASELINE WER = {BASE_WER*100:.2f}%  (if ~80-100% something didn't match -> stop)")

FP32 baseline: 100%|██████████| 98/98 [02:12<00:00,  1.35s/it]


  -> Baseline FP32 (torch)      WER  21.63%  size   216.57 MB
   saved -> /content/drive/MyDrive/silent_speech/compressed/gaddy_fp32.pt (216.6 MB)

BASELINE WER = 21.63%  (if ~80-100% something didn't match -> stop)


In [8]:
# ===== METHOD 1: Dynamic INT8 (weights-only) — CPU only (quantised ops don't run on CUDA) =====
dq = torch.quantization.quantize_dynamic(copy.deepcopy(model).cpu(), {torch.nn.Linear}, dtype=torch.qint8).eval()
log_result("Dynamic INT8 (torch)", wer_torch(dq, device="cpu", desc="dyn-int8"), sd_mb(dq))
save_torch(dq, "gaddy_dynint8.pt")

dyn-int8: 100%|██████████| 98/98 [02:59<00:00,  1.84s/it]


  -> Dynamic INT8 (torch)       WER  21.69%  size   129.79 MB
   saved -> /content/drive/MyDrive/silent_speech/compressed/gaddy_dynint8.pt (129.8 MB)


'/content/drive/MyDrive/silent_speech/compressed/gaddy_dynint8.pt'

In [9]:
# ===== METHOD 2: FP16 half (GPU) =====
if torch.cuda.is_available():
    h = copy.deepcopy(model).half().cuda().eval()
    log_result("FP16 half (torch)", wer_torch(h, device="cuda", half=True, desc="fp16"), sd_mb(h))
    save_torch(h, "gaddy_fp16.pt")
else:
    print("no GPU -> skip FP16")

fp16: 100%|██████████| 98/98 [02:13<00:00,  1.36s/it]


  -> FP16 half (torch)          WER  21.51%  size   108.31 MB
   saved -> /content/drive/MyDrive/silent_speech/compressed/gaddy_fp16.pt (108.3 MB)


In [10]:
# ===== METHOD 3: Magnitude pruning sweep (unstructured global L1) — each saved =====
import torch.nn.utils.prune as prune
def collect_prunable(m):
    tgt=[]
    for mod in m.modules():
        if isinstance(mod,(torch.nn.Conv1d,torch.nn.Linear)): tgt.append((mod,"weight"))
    for name,_ in m.named_parameters():
        if name.split(".")[-1] in ("w_q","w_k","w_v","w_o"):
            owner=m.get_submodule(name.rsplit(".",1)[0]); tgt.append((owner,name.rsplit(".",1)[1]))
    return tgt
for amount in [0.3, 0.5, 0.7]:
    pm=copy.deepcopy(model); tgt=collect_prunable(pm)
    prune.global_unstructured(tgt, prune.L1Unstructured, amount=amount)
    for mod,nm in tgt: prune.remove(mod,nm)
    w=wer_torch(pm, device=DEVICE, desc=f"prune{int(amount*100)}")
    nz=sum((p!=0).sum().item() for p in pm.parameters()); tot=sum(p.numel() for p in pm.parameters())
    log_result(f"Prune {int(amount*100)}% (dense)", w, sd_mb(pm))
    print(f"       sparsity={100*(1-nz/tot):.1f}%  sparse-est~{sd_mb(pm)*nz/tot:.1f} MB")
    save_torch(pm, f"gaddy_prune{int(amount*100)}.pt")
print("NOTE: dense .pt keeps full size; real shrink needs sparse storage or quantisation (next cell).")

prune30: 100%|██████████| 98/98 [02:22<00:00,  1.45s/it]


  -> Prune 30% (dense)          WER  22.48%  size   216.57 MB
       sparsity=29.5%  sparse-est~152.8 MB
   saved -> /content/drive/MyDrive/silent_speech/compressed/gaddy_prune30.pt (216.6 MB)


prune50: 100%|██████████| 98/98 [04:39<00:00,  2.85s/it]


  -> Prune 50% (dense)          WER  63.79%  size   216.57 MB
       sparsity=49.1%  sparse-est~110.2 MB
   saved -> /content/drive/MyDrive/silent_speech/compressed/gaddy_prune50.pt (216.6 MB)


prune70: 100%|██████████| 98/98 [05:49<00:00,  3.57s/it]


  -> Prune 70% (dense)          WER  99.94%  size   216.57 MB
       sparsity=68.7%  sparse-est~67.7 MB
   saved -> /content/drive/MyDrive/silent_speech/compressed/gaddy_prune70.pt (216.6 MB)
NOTE: dense .pt keeps full size; real shrink needs sparse storage or quantisation (next cell).


In [11]:
# ===== METHOD 4: Prune 50% + Dynamic INT8 =====
pm=copy.deepcopy(model); tgt=collect_prunable(pm)
prune.global_unstructured(tgt, prune.L1Unstructured, amount=0.5)
for mod,nm in tgt: prune.remove(mod,nm)
pq=torch.quantization.quantize_dynamic(pm.cpu(), {torch.nn.Linear}, dtype=torch.qint8).eval()
log_result("Prune50% + Dyn-INT8", wer_torch(pq, device="cpu", desc="prune50+int8"), sd_mb(pq))
save_torch(pq, "gaddy_prune50_int8.pt")

prune50+int8: 100%|██████████| 98/98 [05:31<00:00,  3.38s/it]


  -> Prune50% + Dyn-INT8        WER  64.64%  size   129.79 MB
   saved -> /content/drive/MyDrive/silent_speech/compressed/gaddy_prune50_int8.pt (129.8 MB)


'/content/drive/MyDrive/silent_speech/compressed/gaddy_prune50_int8.pt'

In [13]:
# ===== METHOD 5: ONNX (dynamo export, BatchNorm folded) + ORT Dynamic INT8 =====
from onnxruntime.quantization import quantize_dynamic, QuantType
import onnxruntime as ort
try:
    from torch.nn.utils.fusion import fuse_conv_bn_eval
except Exception:
    from torch.nn.utils import fuse_conv_bn_eval

def fuse_for_export(mdl):
    """Fold Conv+BN (exact in eval) so the ONNX exporter has no BatchNorm op to choke on."""
    mm = copy.deepcopy(mdl).eval()
    for rb in mm.modules():
        if isinstance(rb, ResBlock):
            rb.conv1 = fuse_conv_bn_eval(rb.conv1, rb.bn1); rb.bn1 = torch.nn.Identity()
            rb.conv2 = fuse_conv_bn_eval(rb.conv2, rb.bn2); rb.bn2 = torch.nn.Identity()
            if rb.residual_path is not None:
                rb.residual_path = fuse_conv_bn_eval(rb.residual_path, rb.res_norm); rb.res_norm = torch.nn.Identity()
    return mm

class Wrap(torch.nn.Module):
    def __init__(s, m): super().__init__(); s.m = m
    def forward(s, raw):
        b, t, _ = raw.shape; return s.m(raw, raw, torch.zeros(b, t, dtype=torch.long))

FP32_ONNX = "/content/gaddy_fp32.onnx"; INT8_ONNX = "/content/gaddy_int8dyn.onnx"
ex = TEST[0][0].unsqueeze(0)
try:
    fused = fuse_for_export(model)                       # <-- the fix
    prog = torch.onnx.export(Wrap(fused).eval(), (ex,), dynamo=True,
            dynamic_shapes={"raw": {1: torch.export.Dim("t", min=8, max=32000)}})
    prog.save(FP32_ONNX)
    _sess = ort.InferenceSession(FP32_ONNX, providers=["CPUExecutionProvider"]); _in = _sess.get_inputs()[0].name
    with torch.no_grad():
        tref = fused(ex, ex, torch.zeros(1, 1, dtype=torch.long)).numpy()
    oref = _sess.run(None, {_in: ex.numpy()})[0]
    print("ONNX input:", _in, "| parity max|diff| =", float(np.abs(tref - oref).max()))
    quantize_dynamic(FP32_ONNX, INT8_ONNX, weight_type=QuantType.QInt8)
    log_result("ONNX FP32 (non-compressed)", wer_onnx(FP32_ONNX, "onnx-fp32"), file_mb(FP32_ONNX))
    log_result("ONNX Dyn-INT8", wer_onnx(INT8_ONNX, "onnx-int8"), file_mb(INT8_ONNX))
    save_file(FP32_ONNX, "gaddy_fp32.onnx"); save_file(INT8_ONNX, "gaddy_int8dyn.onnx")
except Exception as e:
    print("ONNX export failed on this torch version ->", type(e).__name__, str(e)[:160])
    print("Skipping ONNX methods; your torch results + CSV still complete.")

[torch.onnx] Obtain model graph for `Wrap([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Wrap([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


W0922 13:09:05.109000 234 torch/fx/experimental/symbolic_shapes.py:6184] Ignored guard Ne(Mod(2*s3**2 + s3 - 1, s3 + 1), 0) == False, this could result in accuracy problems
W0922 13:09:11.859000 234 torch/fx/experimental/symbolic_shapes.py:6184] Ignored guard s3 == 408, this could result in accuracy problems


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
ONNX input: raw | parity max|diff| = 4.00543212890625e-05


onnx-fp32: 100%|██████████| 98/98 [11:02<00:00,  6.76s/it]


  -> ONNX FP32 (non-compressed) WER  21.51%  size   222.44 MB


onnx-int8: 100%|██████████| 98/98 [11:44<00:00,  7.19s/it]


  -> ONNX Dyn-INT8              WER  22.30%  size   192.34 MB
   saved -> /content/drive/MyDrive/silent_speech/compressed/gaddy_fp32.onnx (222.4 MB)
   saved -> /content/drive/MyDrive/silent_speech/compressed/gaddy_int8dyn.onnx (192.3 MB)


In [14]:
# ===== METHOD 6: ONNX Static INT8 (QDQ, calibrated). May collapse (like TinyMyo). =====
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantFormat, QuantType
STATIC_ONNX = "/content/gaddy_int8static.onnx"
if not os.path.exists(FP32_ONNX):
    print("no FP32 ONNX (Cell 12 skipped) -> skipping static INT8")
else:
    _nm = ort.InferenceSession(FP32_ONNX, providers=["CPUExecutionProvider"]).get_inputs()[0].name
    class Calib(CalibrationDataReader):
        def __init__(self, n=40):
            self.data = [{_nm: TEST[i][0].unsqueeze(0).numpy().astype(np.float32)} for i in range(min(n, len(TEST)))]
            self.it = iter(self.data)
        def get_next(self): return next(self.it, None)
    try:
        quantize_static(FP32_ONNX, STATIC_ONNX, Calib(40), quant_format=QuantFormat.QDQ,
                        per_channel=True, weight_type=QuantType.QInt8, activation_type=QuantType.QInt8)
        log_result("ONNX Static-INT8", wer_onnx(STATIC_ONNX, "onnx-static"), file_mb(STATIC_ONNX))
        save_file(STATIC_ONNX, "gaddy_int8static.onnx")
    except Exception as e:
        print("static INT8 not supported for this export ->", type(e).__name__, str(e)[:120])

onnx-static: 100%|██████████| 98/98 [15:52<00:00,  9.72s/it]


  -> ONNX Static-INT8           WER 100.00%  size   189.67 MB
   saved -> /content/drive/MyDrive/silent_speech/compressed/gaddy_int8static.onnx (189.7 MB)


In [19]:
# ===== METHOD 7: Knowledge Distillation (GPU; teacher logits cached in RAM). Set RUN_KD=True. =====
RUN_KD = True
if RUN_KD:
    dev = DEVICE
    teacher = copy.deepcopy(model).eval().to(dev)
    for p in teacher.parameters(): p.requires_grad_(False)
    trainset = H5EmgDataset(dev=False, test=False)
    print("caching teacher logits for", len(trainset), "utterances (uses RAM) ...")
    CACHE = []   # (raw_cpu, teacher_logsoftmax_cpu, target)
    T = 2.0; dummy = torch.zeros(1,1,dtype=torch.long,device=dev)
    with torch.no_grad():
        for i in tqdm.tqdm(range(len(trainset))):
            ex = trainset[i]; tgt = trainset.text_transform.text_to_int(ex["text"])
            if len(tgt)==0: continue
            raw = ex["raw_emg"].unsqueeze(0).to(dev)
            tl = F.log_softmax(teacher(raw, raw, dummy)/T, dim=-1).cpu()
            CACHE.append((ex["raw_emg"], tl, torch.tensor(tgt)))
    student = Model(0, NUM_OUTS, model_size=256, num_layers=3).train().to(dev)
    opt = torch.optim.AdamW(student.parameters(), lr=3e-4)
    ALPHA, BLANK, EPOCHS = 0.7, N_CHARS, 20
    for ep in range(EPOCHS):
        tot=0.0
        for idx in torch.randperm(len(CACHE)):
            raw, tl, tgt = CACHE[int(idx)]
            raw=raw.unsqueeze(0).to(dev); tl=tl.to(dev); tgt=tgt.to(dev)
            s_out = student(raw, raw, dummy)
            kd = F.kl_div(F.log_softmax(s_out/T,dim=-1), tl, log_target=True, reduction="batchmean")*(T*T)
            ctc = F.ctc_loss(F.log_softmax(s_out,-1).transpose(0,1), tgt,
                             torch.tensor([s_out.shape[1]],device=dev), torch.tensor([tgt.numel()],device=dev),
                             blank=BLANK, zero_infinity=True)
            loss = ALPHA*kd + (1-ALPHA)*ctc
            opt.zero_grad(); loss.backward(); opt.step(); tot+=loss.item()
        print(f"epoch {ep+1}/{EPOCHS}  mean loss {tot/len(CACHE):.3f}")
    student.eval()
    log_result("KD student", wer_torch(student, device=dev, desc="kd-student"), sd_mb(student))
    save_torch(student, "gaddy_kd_student.pt")
else:
    print("KD skipped. Set RUN_KD=True to train (GPU, ~minutes-hours). Thesis: KD from this teacher fails (~80%).")

caching teacher logits for 1289 utterances (uses RAM) ...


100%|██████████| 1289/1289 [00:26<00:00, 48.75it/s]


epoch 1/20  mean loss 2067.995
epoch 2/20  mean loss 1681.578
epoch 3/20  mean loss 1503.887
epoch 4/20  mean loss 1380.159
epoch 5/20  mean loss 1271.895
epoch 6/20  mean loss 1177.062
epoch 7/20  mean loss 1091.986
epoch 8/20  mean loss 1010.670
epoch 9/20  mean loss 935.890
epoch 10/20  mean loss 860.559
epoch 11/20  mean loss 798.828
epoch 12/20  mean loss 728.243
epoch 13/20  mean loss 679.965
epoch 14/20  mean loss 628.413
epoch 15/20  mean loss 584.276
epoch 16/20  mean loss 548.416
epoch 17/20  mean loss 520.476
epoch 18/20  mean loss 483.372
epoch 19/20  mean loss 473.908
epoch 20/20  mean loss 427.845


kd-student: 100%|██████████| 98/98 [03:43<00:00,  2.28s/it]

  -> KD student                 WER  67.68%  size    27.55 MB
   saved -> /content/drive/MyDrive/silent_speech/compressed/gaddy_kd_student.pt (27.6 MB)


In [20]:
# ===== SUMMARY: save CSV to Drive + print it =====
import pandas as pd
df = pd.DataFrame(RESULTS)
df["reduction_x"] = (BASE_SIZE / df["size_MB"]).round(2)
df["within_10%_rel"] = df["WER%"] <= round(BASE_WER*100*1.10, 2)
CSV = f"{DRIVE_PROJECT}/gaddy_compression_results.csv"
df.to_csv(CSV, index=False)
print("saved CSV ->", CSV)
print("\nsaved models in", OUT_DIR, ":")
for f in sorted(os.listdir(OUT_DIR)): print("   ", f, f"({os.path.getsize(os.path.join(OUT_DIR,f))/1e6:.1f} MB)")
print("\n===== RESULTS (read back from CSV) =====")
print(pd.read_csv(CSV).to_string(index=False))

saved CSV -> /content/drive/MyDrive/silent_speech/gaddy_compression_results.csv

saved models in /content/drive/MyDrive/silent_speech/compressed :
    gaddy_dynint8.pt (129.8 MB)
    gaddy_fp16.pt (108.3 MB)
    gaddy_fp32.onnx (222.4 MB)
    gaddy_fp32.pt (216.6 MB)
    gaddy_int8dyn.onnx (192.3 MB)
    gaddy_int8static.onnx (189.7 MB)
    gaddy_kd_student.pt (27.6 MB)
    gaddy_prune30.pt (216.6 MB)
    gaddy_prune50.pt (216.6 MB)
    gaddy_prune50_int8.pt (129.8 MB)
    gaddy_prune70.pt (216.6 MB)

===== RESULTS (read back from CSV) =====
                    method   WER%  size_MB  reduction_x  within_10%_rel
     Baseline FP32 (torch)  21.63   216.57         1.00            True
      Dynamic INT8 (torch)  21.69   129.79         1.67            True
         FP16 half (torch)  21.51   108.31         2.00            True
         Prune 30% (dense)  22.48   216.57         1.00            True
         Prune 50% (dense)  63.79   216.57         1.00           False
         Prune 70% (